In [1]:
#General Dependencies
from pathlib import Path
import shutil, os, glob, re
import pandas as pd
import numpy as np
from natsort import natsorted
from file_read_backwards import FileReadBackwards
import filecmp

"""

          __      _.._       ___        _.._               _     
       .-'__`-._.'.--.'.__.-'.--.`-._.''.--. '._.,        | | 
      /--'  '-._.'    '-._.'' __ '-._.' ._.  '._./        | |__   __ _  ___ ___  _ __ 
     /__.--._.--._.'``-.__.''`  '`-.__.'   '--._/         | '_ \ / _` |/ __/ _ \| '_ \ 
     '._.-'-._.-._.-''-..'._.-''-..''-..''-.__.'          | |_) | (_| | (_| (_) | | | | 
             @GCH v1.0 - last updt. 03/23/2026            |_.__/ \__,_|\___\___/|_| |_|

### Notebook Overview:
This notebook contains code to process ORCA 5.0.3 output (.out) files. 'Successful' jobs correspond to converged optimized geometries
with zero imaginary frequencies. 'Failed Jobs' are triaged based on common calculation outcomes, ex: failed convergence, failed to
terminate, presence of imaginary frequences. 

### Planned Features:
1. Add failure-specific .inp creation for resubmission of any failed calculations.
   - Non-terminated jobs (failed due to wall-time constraints)
   - Non-converged geometries (resubmit with an inferred/derived "best" geometry)
   - Imaginary Frequencies (perturb along individual IFs; derive new .inp)
2. Check that there doesn't need to be an explicit check for convergence/add one anyway for sanity/robustness
3. ...
...

### Motivation:
Replace the use of QCORR with something substantially faster/more transparent/customizable. Can be readily modified for Orca6.

### How to Use this Notebook: 
1. Define relevant Paths to output files in the PATH cell below
2. Download all .out files from an HPC to the relevant output paths (see example, below)
3. Run the notebook from the top-level project directory to process/sort .out files
4. There are also cells below to regenerate new .inp files from failed/non-converged first-run calculations

#### Example:
I have 1 dir for aryne .outs (/project_dir/aryne_ORCA_ DFT_Outputs/) and 1 dir for Aryne .outs (/project_dir/aryne_ORCA_ DFT_Outputs/). I define paths in the ### PATHS ### cell below to indicate that these two dirs contain my .out files that I want to process. I run the cells below on each of my two target directories. After, each dir (arynes/arynes) will contain validate '.outs' and a subdir called /failed_calculations. Within /failed_calculations, there are subdirs for jobs that failed due to different reasons.

### How Bacon processes .out files:
1. First pass, the 'sort_complete_fail_jobs_orca5()' method sorts jobs according to the ending text in the .out file:
    - 'Normal Termination' - either completed calcs or those that ended in failed convergence
        - Normal termination/no errors - stay in top level dir/don't move file
        - Normal termination/failed convergence - move .outs to /project_dir/Aryne_ORCA_ DFT_Outputs/convergence_failure/ 
    - Failed/No 'Normal Termination' => sent to /project_dir/Aryne_ORCA_ DFT_Outputs/failed_calculations" for manual
        triage - usually these are just genuinely bad calculations


2. Second pass evaluates the putatively "good" calculations for convergence and performs an imaginary frequency check.
    - does_out_contain_freq_data() method checks the 'normal termination' jobs for text flags indicating a frequency calc. was
        performed
    - get_orca5_freqs() method extracts the last freq block from the .out file 
    - has_imaginary_freqs_check() method analyzes extracted freqs for imaginary values

End result should be a top-level dir containing validated .out files and subdirs containing calcs that failed due to various
reasons. Can resubmit these jobs as-needed and re-process.

"""

In [2]:
### Define PATHS/relevant directories ### 
project_dir = Path.cwd() #top level project dir

### ARYNES Top Level Job Directory and triage subdirs ### 
aryne_dft_output_path = project_dir/"DFT_Aryne_Data"/"Aryne_Orca_DFT_Outputs" #aryne_outs

#dir containing .inp files for referencing expected .out files 
aryne_dft_input_path = project_dir/"DFT_Aryne_Data"/"Aryne_Orca_DFT_Inputs"

#dir for writing new .inp files to for resubmission
aryne_dft_resub_input = aryne_dft_input_path/"Aryne_Resubs"
aryne_dft_resub_input.mkdir(exist_ok=True)

#dir for jobs that failed due to bad failures/didn't terminate
aryne_failed_to_terminate = aryne_dft_output_path/"Failed_Calculations"
aryne_failed_to_terminate.mkdir(exist_ok=True)

#dir for jobs that failed due to convergence failure
aryne_no_convergence_dir = aryne_failed_to_terminate /"Convergence_Failure"
aryne_no_convergence_dir.mkdir(exist_ok=True)

#dir for jobs that failed due to having an extra (or more) imaginary frequency
aryne_im_freq_dir= aryne_failed_to_terminate/"Extra_Imaginary_Frequency"
aryne_im_freq_dir.mkdir(exist_ok=True)

In [3]:
def get_filepaths_in_target_dir(target_directory: Path, extension: str, printing=True):
    """
    Function: returns a list of Path objects pertaining to files in a specified directory. Searches for specific .ext
    
    Input:
        - target_directory; Path containing files with the '.extension'
        - extension; str: a string specifying an extension to use, ex: ".out", ".inp", ".sdf" etc.
        - printing; bool; Default value = True; controls printing back to user
        
    Returns: 
        - filepaths - a list of filepaths for each file wit .extension' in 'target_directory'
    """
    
    #grab user extension with a search wildcard
    file_extension = f"*{extension}"
    
    #gather and sort .ext files in /target_directory/
    filepaths = target_directory.glob(file_extension)
    filepaths = natsorted(filepaths, key=str)

    #return some user info
    if printing:
        print(f"\033[1mFound {len(filepaths)} {extension} files in '/{target_directory.stem}/\033[0m'")

    #return the list of located filepaths
    return filepaths

In [4]:
def get_filepaths_in_target_dir_recursive(target_directory: Path, extension: str, printing=True):
    """
    Function: returns a list of Path objects pertaining to files in a specified directory. Searches recursively
        for files matching the specified extension.
    
    Input:
        - target_directory; Path containing files with the '.extension'
        - extension; str: a string specifying an extension to use, ex: ".out", ".inp", ".sdf" etc.
        - printing; bool; Default value = True; controls printing back to user
        
    Returns: 
        - filepaths - a list of filepaths (recursive) for each file wit .extension' in 'target_directory' 
    """
    
    #grab user extension with a search wildcard
    file_extension = f"*{extension}"
    
    #gather and sort .ext files in /target_directory/
    filepaths = target_directory.rglob(file_extension)
    filepaths = natsorted(filepaths, key=str)

    #return some user info
    if printing:
        print(f"\033[1mFound {len(filepaths)} {extension} files in '/{target_directory.stem}/' (recursive)\033[0m")

    #return the list of located filepaths
    return filepaths

In [5]:
def sort_complete_fail_jobs_orca5(path_to_out_file: Path):
    """
    Function: Given an .out file, determine if it completed successfully or terminated with an error.
        Triages DFT .out files based on their success/failure type. 
    
    Input:
        - path_to_out_file: Path; a Path specifying a DFT .out file
        
    Returns: 
        - filepaths - a list of filepaths for each file wit .extension' in 'target_directory'
    """

    #local capture of some vars/name vars
    file_path = path_to_out_file
    parent_dir = path_to_out_file.parent

    #Strings that indicate specific job fate; used for triaging DFT .outs according to common failure modes
    normal_termination_str = "****ORCA TERMINATED NORMALLY****" #these are potentially good calculations
    failed_convergence_str = "The optimization did not converge but reached the maximum" #these failed convergence during opt
    #Store all of the flags as a list to check against
    ending_flags = [normal_termination_str, failed_convergence_str]

    #read each .out file starting from the end and working up line-by-line
    try:
        with FileReadBackwards(file_path, encoding="utf-8") as file: #read backwards from end for speed/mem
            #boolean for whether or not a known termination flag is in a line
            flag_found = False

            #read just the last 20 or so lines of an .out file for speed/mem - searching for ending flags only
            for i in range(19):
                line = file.readline()

                #out file should have at least 20 lines to be even sort of feasibly OK
                if not line:
                    print(f"'{file_path.name}': Calculation failed catastrophically; check by hand")
                    break

                #If the first 20 or so lines contain one of the known ending flags, job terminated somehow (known outcome)
                if any(ending_flag in line for ending_flag in ending_flags):
                    flag_found = True #one of the known ending flags was found; triage the type of termination

                    #most common in an ideal world; should be first condition checked
                    #these jobs terminated and should be evaluated for quality after this first pass
                    #need to check them for imfreqs and for convergence criteria
                    if normal_termination_str in line:
                        print(f"\t'{file_path.name}': Potentially Valid")
                        #dont do anything with it - you can uncomment below if you want to do something
                        # #move the file to /completed_jobs_dir/
                        # file.close() #manually close the file before moving
                        # destination_file_path = completed_jobs_dir / file_path.name
                        # try:
                        #     shutil.move(file_path, destination_file_path)
                        #     print(f"'{file_path.name}': \tNormal Termination. Moved to /normal_termination/ for validation")
                        # except FileNotFoundError:
                        #     print(f"Error: Source file not found at {file_path}")
                        # except shutil.SameFileError:
                        #     print(f"Skipped '{file_path.name}', already exists in destination.")
                        # except Exception as e:
                        #     print(f"Error copying {file_path.name}: {e}")
                        break

                    #jobs that finished but didn't converge to opt'd geometry
                    elif failed_convergence_str in line: 
                        #move the file to /no_convergence_dir/
                        file.close()
                        destination_file_path = parent_dir/"failed_calculations"/"convergence_failure"/file_path.name
                        try:
                            shutil.move(file_path, destination_file_path)
                            print(f"'{file_path.name}': \tFailed Convergence. Moved to /convergence_failure/ for resub")
                        except FileNotFoundError:
                            print(f"Failed Conv Error - Error: Source file not found at {file_path}")
                        except shutil.SameFileError:
                            print(f"Skipped '{file_path.name}', already exists in destination.")
                        except Exception as e:
                            print(f"Error copying {file_path.name}: {e}")
                        break
                        
                        #want to make new input files for these jobs using the final geometry from the output file
                        #should check if the geometry still makes any sense (is this fast enough? idk if it's worth)
                        #find the last geometry in the output
                        #make a new input
                        #look at the convergence settings tab in orca also - maybe some things I can implement to improve chances in harder ones
                        break
                    break

            #if none of the known termination flags are found in the last 20 or so lines, job did not finish
            if not flag_found:
                file.close()
                destination_file_path = parent_dir/"failed_calculations"/file_path.name
                try:
                    shutil.move(file_path, destination_file_path)
                    print(f"'{file_path.name}': \tDid not terminate. Moved to /failed_calculations/ for further triage.")
                except FileNotFoundError:
                    print(f"Error: Source file not found at {file_path}")
                except shutil.SameFileError:
                    print(f"Skipped '{file_path.name}', already exists in destination.")
                except Exception as e:
                    print(f"Error copying {file_path.name}: {e}")
                #move them to a new dir or make a new resub input
                #kind of depends on what happened with them; can maybe trirage more
                    
    except FileNotFoundError:
        print(f"'{file_path}' does not exist.")
        
    except PermissionError:
        print(f"Permission denied for '{file_path}'.")

In [6]:
def does_out_contain_freq_data(path_to_out_file: Path):
    """
    Purpose: Check an Orca 5.x.x (tested Orca 5.0.3) .out file for evidence that a 'freq' calc occurred
    
    Input:
        - A path to an Orca 5.x.x .out file
        
    Returns:
        - True: if .out file contains at least one block of frequency data
            #Note: it is hard coded to look for a full start-end sequence;
            #partial "freq" blocks will fail the check
        - False: if .out file does not contain any complete frequency block
    """

    #some path things
    file_path = path_to_out_file
    parent_dir = file_path.parent
    no_freq_dir = parent_dir/"failed_calculations"/"no_freq_data"
    no_freq_dir.mkdir(exist_ok=True) #make dirs if they dont exist

    #Set this to whatever text signifier you want; 
    #since we are reading from the back to the front, the first thing we should hit
    #in an Orca5 output is "NORMAL MODES" that terminates the freq block in an .out
    orca_freq_line = "NORMAL MODES"

    #read through the file backwards for our desired text line
    try:
        with FileReadBackwards(file_path, encoding="utf-8") as file:
            for line in file: #remember we are going backwards

                #if the line indicating Freq calc was performed is found, 
                if orca_freq_line in line:
                    #print(f'{file_path.name} contains freq data!')
                    return True 
                    break #exit as soon as a match is 
                    
            else:  #no frequency data was found
                print(f'{file_path.name} does not contain frequency data.')
                
                #move the file to an error dir /no_freq_data/
                file.close() #make sure to close the file first
                destination_file_path = no_freq_dir / file_path.name

                try:
                    shutil.move(file_path, destination_file_path)
                    print(f"'{file_path.name}': \tDid not terminate. Moved to /failed_calculations/ for further triage.")
                except FileNotFoundError:
                    print(f"Error: Source file not found at {file_path}")
                except shutil.SameFileError:
                    print(f"Skipped '{file_path.name}', already exists in destination.")
                except Exception as e:
                    print(f"Error copying {file_path.name}: {e}")
                return False
                
    except FileNotFoundError:
        print(f"'{file_path.name}' does not exist.")
        
    except PermissionError:
        print(f"Permission denied for '{file_path.name}'.")

In [7]:
def get_orca5_freqs(path_to_out_file: Path):
    """
    Function: Manually captures the vibrational frequencies from an ORCA 5.0.3 output file using string matching
        and regex.

    Input:
        - path_to_out_file: Path; Path to an ORCA .out file

    Returns:
        - frequencies: List; A list of extracted frequencies in cm^-1
    """

    #local capture
    file_path = path_to_out_file

    frequencies = [] #list to store captured frequency data
    freq_flag = False #This flag is True when lines are within the "Frequency" block of the .log file

    #read the file from back to front to find the freq data block in .out (faster back to front, generally)
    try:
        with FileReadBackwards(file_path, encoding="utf-8") as file:
            freq_found_in_file = False #we want to see if the file contains ANY freq data first
            for line in file: #remember we are going backwards
            
                if "NORMAL MODES" in line: #this is the BOTTOM of the freq 
                    freq_found_in_file = True #Set flag to True indicating that ANY freq data was found
                    freq_flag = True #fly it
                    #print(f"Found the bottom of the freq section in: {line.strip()}") #debug
                    continue

                if freq_flag: #if the freq line has been found, we need to capture lines of text
                    stripped_line = line.strip() #strip the line of text of weird whitespace
                        
                    if re.match(r"^\s*\d", stripped_line): #looking for lines that start with no whitespace and a digit
                        split_elements = stripped_line.split() #split the matched line
                        frequencies.append(float(split_elements[1])) #store the freq data as type: float
                        
                    elif "Scaling factor for frequencies" in line: #this is the TOP of the freq block
                        break #we want to stop capturing lines if the top of the block is hit

        #check if the file has freq data
        if freq_found_in_file: 
            return frequencies[::-1] #if there is, give the reverse of the captured list (we read back to front)
        else:
            return("No Frequency data was found in your file") #throw an error if there's no freq data

    except FileNotFoundError:
        print(f"'{file_path}' does not exist.")
        
    except PermissionError:
        print(f"Permission denied for '{file_path}'.")
      

In [8]:
def has_imaginary_freqs_check(list_of_freqs: list): 
    """
    Function: Checks a list of normal vibrational modes for any imaginary (in this case <0) frequencies
    
    Input: 
        - A list of frequencies extracted with the 'get_orca5_freqs()' method
        
    Returns: 
        'True' if 1 or more imaginary frequencies were found (fails the check)
        'False' if all normal modes are positive
    """
    
    return any(x < 0 for x in list_of_freqs)

In [9]:
def how_many_jobs_where(path_to_target_directory: Path):
    """
    Function: Checks a list of normal vibrational modes for any imaginary (in this case <0) frequencies
    
    Input: 
        - A list of frequencies extracted with the 'get_orca5_freqs()' method
        
    Returns: 
        'True' if 1 or more imaginary frequencies were found (fails the check)
        'False' if all normal modes are positive
    """
    target_dir = path_to_target_directory
    print(f'\nChecking {target_dir} for Orca.out files after processesing...')

    #if the code above was used to sort the jobs, these dirs should exist (if there were failed calcs)
    failed_to_terminate_dir = target_dir/"Failed_Calculations"
    no_convergence_dir = failed_to_terminate_dir/"Convergence_Failure"
    extra_imag_freq_dir = failed_to_terminate_dir/"Extra_Imaginary_Frequency"
    no_freq_data = failed_to_terminate_dir/"no_freq_data"

    #how many successful jobs in the target dir (no subdirs)
    successful_outs = get_filepaths_in_target_dir(target_dir, ".out", printing=False)
    print(f'\n\033[1mNumber of VALIDATED ORCA.out files in {target_dir.name}: {len(successful_outs)}\033[0m')

    #if failed dirs exist, check other dirs for jobs/count the failed jobs in each
    if failed_to_terminate_dir.is_dir():
        all_failed_jobs = get_filepaths_in_target_dir_recursive(failed_to_terminate_dir, ".out", printing=False)
        print(f'\n\033[1mNumber of FAILED ORCA.outs in /{failed_to_terminate_dir.name}/: {len(all_failed_jobs):>12}\033[0m')
        
        jobs_with_ifs = get_filepaths_in_target_dir(extra_imag_freq_dir, ".out", printing=False)
        num_with_ifs = len(jobs_with_ifs)
        print(f'\t- Failed due to extraneous Imaginary Freqs: {num_with_ifs:>13}')

        no_conv_jobs = get_filepaths_in_target_dir(no_convergence_dir, ".out", printing=False)
        num_no_conv = len(no_conv_jobs)
        print(f'\t- Failed due to lack of convergence: {num_no_conv:>20}')

        failed_disasters = get_filepaths_in_target_dir(failed_to_terminate_dir, ".out", printing=False)
        num_total_fails = len(failed_disasters)
        print(f'\t- Complete and Utter Hopeless Failures: {num_total_fails:>17}')

        no_freq_jobs = get_filepaths_in_target_dir(no_freq_data, ".out", printing=False)
        num_no_freq = len(no_freq_jobs)
        print(f'\t- Failed due to missing freq data: {num_no_freq:>22}')
        
    if [len(successful_outs) >= 1] and [len(all_failed_jobs) >= 1]:
        total_outs = int(len(successful_outs) + int(len(all_failed_jobs)))
        print(f'\n\033[1mTotal number of ORCA.out files in /{target_dir.name}/ accounted for: {total_outs}\033[0m')

In [10]:
def get_input_filepaths_in_target_dir_recursive(target_directory: Path):
    """
    Input: Path 'target_directory' containing files with the '.out' extension (DFT output files)
    Returns: a list of filepaths for each file with '.out' extension in 'target_directory'

    could also modify this for any file extension
    """

    #gather and sort output files in target_directory by #
    #note "rglob"
    dft_input_filepaths = target_directory.rglob('*.inp')
    dft_input_filepaths = natsorted(dft_input_filepaths)

    return list(dft_input_filepaths)

In [11]:
def get_mol_ids_from_filepaths(filepaths: Path):
    """
    This chops each filename down into just its integer number
    #ex: aryne_4_rdkit_conf_1.out ==> 4 as type int
    """

    mol_ids = []
    
    for filepath in filepaths:
        file_name = filepath.stem
        file_name_parts = str(file_name).split('_')
        mol_num = f"{file_name_parts[1]}"
        mol_ids.append(int(mol_num))

    return mol_ids

In [12]:
def find_missing(list: list):
    ### fix this nonsense and add documentation
    max = int(list[0])
    for i in list:
        if i > max:
            max = i

    min = int(list[0])
    for i in list:
        if i < min:
            min = i

    list1 = []

    for num in range(min + 1, max):
        if num not in list:
            list1.append(num)

    return list1

In [13]:
def search_and_copy_files(source_dir: Path, dest_dir: Path, filenames_list: list):
    """
    Recursively searches for specified filenames in source_dir and copies them to dest_dir.

    :param source_dir: The root directory to start the search (pathlib.Path object or string).
    :param dest_dir: The destination directory for copied files (pathlib.Path object or string).
    :param filenames_list: A list of filenames (strings) to search for.
    """
    source_path = Path(source_dir)
    dest_path = Path(dest_dir)

    # Create the destination directory if it does not exist
    dest_path.mkdir(parents=True, exist_ok=True)

    # Use rglob to find all files recursively
    # The pattern "*" will match every file, which we then filter by name
    for file_path in source_path.rglob("*"):
        if file_path.is_file() and file_path.name in filenames_list:
            destination_file_path = dest_path / file_path.name
            try:
                # Copy the file to the destination directory
                shutil.copy(file_path, destination_file_path)
                print(f"Copied '{file_path.relative_to(source_path).as_posix()}' to '{destination_file_path.relative_to(source_path).as_posix()}'")
            except shutil.SameFileError:
                print(f"Skipped '{file_path.name}', already exists in destination.")
            except Exception as e:
                print(f"Error copying {file_path.name}: {e}")

In [14]:
def split_dir_into_subdirs(parent_directory_path: Path, num_jobs_in_each_subdir: int):
    """
    Input:
        1. parent_directory_path: path to the directory containing unsorted DFT input files
        2. num_jobs_in_each_subdir: int value for the number of jobs desired in each subdir
    Function:
        1. Creates 'n' subdirs in parent directory dep. on the # of files in parent directory/'num_jobs_in_each_subdir'
        2. moves 'num_jobs_in_each_subdir'-defined number of DFT inputs to each created subdir
    """
    
    #find and sort input files 
    inp_files = parent_directory_path.glob('*.inp')
    inp_files = natsorted(inp_files, key=str)
    print(f'\033[1mFound {len(inp_files)} DFT Orca.inp files in {parent_directory_path}\033[0m')

    ##debug
    # for inp_file in inp_files:
    #     inp_name = inp_file.name
    #     print(inp_name)

    #Assigns 'increment' number of DFT jobs to each subdir
    increment = num_jobs_in_each_subdir
    #keeps track of how many subdirs are created
    subdir_count = 0

    for i in range(0, len(inp_files), increment):

        #grab the filename's header to name a subdir with
        file_path = inp_files[i].stem
        #grab the first word of the filename, in this case either "aryne" or "aryne"
        mol_header = str(file_path).split('_', 1)[0]
        
        #formatting the names of the subdirs to be 'aryne/aryne_inputs_X-Y' 
        #where X and Y are i and i+increment chunks
        subdir = "{}_{}_{}".format(mol_header+"_inputs", i + 1, i + increment)
    
        #make a subdir named 'job_name_i_(i+increment)' to hold 'increment' # of .inps
        target_subdir = parent_directory_path/subdir
        target_subdir.mkdir(parents=True, exist_ok=True)

        for file in inp_files[i:i+increment]:
            ##move the files to target_subdir
            file.rename(target_subdir/file.name)

In [15]:
def write_resub_inp_for_failed_conv(path_to_out_file: Path, path_to_resub_dir: Path):
    """
    Function:
    Inputs:
        -path_to_out_file
        -path_to_resub_dir
    Returns: 
        - 
    """
    
    #capture file name for writing an out file
    file_name = path_to_out_file.stem
    
    #get the job input using get_orca5_inp_from_out()
    inp_from_out = get_orca5_inp_from_out(path_to_out_file)

    #Rip out the current xyz coordinates from captured input:
    stripped_block = strip_geometry_from_inp_orca5(inp_from_out)

    # get the "best" (last - can refine this a bit at some point) geometry from the .out file
    extracted_g =  get_last_geom(path_to_out_file)
    df_str = extracted_g.astype(str)
    extracted_xyz = [' '.join(row) for row in df_str.values]

    #insert the new geometry into the input template
    new_inp = splice_new_geom_into_inp(stripped_block, extracted_xyz)
    #have to flatten the list into a \n-sep string for writing - annoying
    new_inp_as_str = '\n'.join(new_inp)
    new_inp_as_str += '\n'

    out_file_name = file_name+".inp"
    write_path = path_to_resub_dir / out_file_name
    write_path.parent.mkdir(parents=True, exist_ok=True) #make sure the dir exists
    write_path.write_text(new_inp_as_str, encoding='utf-8') #write the new .inp file
    print(f'New inp for resubmission of {file_name} written to /{path_to_resub_dir.stem}/')

    

In [16]:
def get_orca5_inp_from_out(path_to_out_file: Path):
    """
    Manually captures the input parameters from an ORCA 5.0.3 output file
    using string matching and regex.

    Input:
        - path_to_out_file: Path to an ORCA .out file

    Returns:
        - captured_input: a list containing lines of text corresponding to the input block of an orca calc.
        (used as a template to remake new .inp files from .out triage)
    """

    file_path = path_to_out_file

    captured_input = []
    inp_flag = False #This flag is True when lines are within the "Frequency" block of the .log file

    try:
        with open(file_path, 'r') as file:
            for line in file:
            
                if "INPUT FILE" in line: #this is the top of the input block 
                    inp_flag = True #input data has been found/turn on text capture
                    continue

                if inp_flag: # text capture has been turned on
                    stripped_line = line.strip()
                    #add the captured line to our captured input 
                    captured_input.append(stripped_line)
                        
                    if "END OF INPUT" in line: #this is the BOTTOM of the inp block
                        break

        if captured_input: 
            raw_inp_block = captured_input[3:-1]

            formatted_inp = []
            for line in raw_inp_block:
                stripped_line = line.strip()
                split_line = stripped_line.split('>', 1)
                inp_line = split_line[1]
                formatted_inp.append(inp_line)
            return formatted_inp
        else:
            return("No input data was found in your file")

    except FileNotFoundError:
        print(f"'{file_path}' does not exist.")
        
    except PermissionError:
        print(f"Permission denied for '{file_path}'.")
      

In [17]:
def get_last_geom(dft_output_file_path: Path):
    """
    Input: A filepath to a DFT output file (orca)
    """

    file_name = dft_output_file_path

    captured_geom_blocks = []

    #empty string to store captured geom lines
    geom_block = []
    geom_block_list = []
    
    #regex match pattern
    END_PATTERN = '^-'
    
    #turn off printing as default
    printing = False
    found_start = False

    #read through a file line-by-line
    with open(file_name, "r") as file:
        for line in file:
            #Read through lines until hitting the Final Geometry Block indicator in text
            if line.strip().startswith("CARTESIAN COORDINATES (ANGSTROEM)"):
                #once the line has been found, indicate that it's time to start printing lines
                found_start = True
                #stops the current loop iteration and goes to the next line
                continue
    
            #once a loop iter hits the end pattern, do one of two things:	
            elif re.match(END_PATTERN, line):
    
                #if a second end (the last '---' is hit, stop printing. 
                if printing:
                    found_start = False
                    printing = False
                    #get the current geom block and save it to a list
                    geom_block_list.append(geom_block)
                    #empty the geom block so the next can be captured
                    geom_block = []
                    continue
    
                #if a start has been found, capture lines
                #this skips the first --- right under the start line
                if found_start:
                    printing = True
                continue
    
            #if match is found, printing is "on"
            #print/capture the line
            if printing:
                geom_block.append(line.strip())
    
    #the last geometry in this list should be the optimized one
    #can confirm looking for "*** FINAL ENERGY EVALUATION AT THE STATIONARY POINT ***" line
    optimized_geom_raw = geom_block_list[-1]
    final_geom = []
    for line in optimized_geom_raw[:-1]:
        stripped = line.strip()
        geom_line = stripped.split()
        final_geom.append(geom_line)
        
    #convert the geometry to a pd DF
    geom_df = pd.DataFrame(final_geom, columns=["AtomID", "x", "y", "z"])
    #set the index to be from 1 rather than 0 - we will work from atom index indices
    #geom_df.index = pd.RangeIndex(1, len(geom_df.index) + 1)
    
    #note the atom indices are 0, not 1
    return geom_df

In [18]:
def strip_geometry_from_inp_orca5(input_file_as_list: list):
    """
    Function: Remove the .xyz coordinates in the * xyz i j * block of an orca iput file
    Input:
        - a list of lines corresponding to an input file
    Returns:
        - a list of the inp with the xyz coordinates removed
    """

    #Get the list index where the geom block starts explicitly (by line index) - avoid incorrectly
    #grabbing other *input* blocks this way, ex: solvent blocks/nbo blocks etc.
    geom_start_indx = [i for i, inp_line in enumerate(input_file_as_list) if r"* xyz" in inp_line][0]

    #error catch flag - if this is still -1 after looping, no second * was found (unlikely but robust)
    geom_end_indx = -1

    #starting at the geom start index, find the index of the next * in the input lines
    for i in range(geom_start_indx +1, len(input_file_as_list)):
        if input_file_as_list[i].strip().startswith('*'):
            geom_end_indx = int(i)

    start_del = geom_start_indx + 1
    end_del = geom_end_indx 
    # #confirm we've found the correct numbers
    # print(f"Target for deletion: {start_del}-{end_del}")

    #Remove lines between the two captured indices (should be just the geometry in .xyz's 
    chopped_input = input_file_as_list[:start_del] + input_file_as_list[end_del:]

    return chopped_input


In [19]:
def splice_new_geom_into_inp(list_of_lines_for_input: list, new_xyz_coords: list):
    """
    Function: 
    Input: 
        - list_of_lines_for_input: list - 
        - new_xyz_coords: list - 
    Returns:
        -
    """
    
    input_lines = list_of_lines_for_input
    xyz_to_insert = new_xyz_coords

    #find the list index of the input lines
    geom_start_indx = [i for i, inp_line in enumerate(input_lines) if r"* xyz" in inp_line][0]
    #we will start inserting our geom lines
    insert_indx = geom_start_indx + 1

    list_of_lines_for_input[insert_indx:insert_indx] = xyz_to_insert

    return list_of_lines_for_input  

In [20]:
def extract_displacement_vectors_as_df(output_file_path: Path):
    """
    Extracts normal mode displacement vectors from an ORCA output file.
    """
    #capture the block of text between 'NORMAL MODES' and 'IR SPECTRUM'
    #this block contains the Normalized displacement vectors corresponding to 
    #individual normal vibrational modes - used to displace geometries along
    #specific modes

    displacement_vectors_raw = [] #capture the raw text for subsequent parsing
    collecting_displacements = False #start with capture "off" until start marker

    try:
        with FileReadBackwards(output_file_path, encoding="utf-8") as file:
        #with open(output_file_path, 'r') as f:
            for line in file:
                # Check for the main normal mode header
                if r"IR SPECTRUM" in line:
                    collecting_displacements = True
                    continue

                if collecting_displacements:
                    stripped_line = line.strip()

                    #just capture stripped lines that start with a #
                    if re.match(r"^\s*\d", stripped_line): 
                        displacement_vectors_raw.append(stripped_line) #capture the raw text

                    elif "NORMAL MODES" in line: #this is the TOP of the displacements block
                        break

    except FileNotFoundError:
        print(f"Error: The file {output_file_path} was not found.")
        return None
        
    except Exception as e:
        print(f"An error occurred during parsing: {e}")
        return None

    #reverse the order of the captured raw text
    displacement_vectors_raw.reverse()

    sections = []
    current_section = []
    
    #break the raw output block into sections of 6 displacements:
    for line in displacement_vectors_raw:
        split_line = line.split()
        if not split_line:
            continue
            
        num_elem_in_line = len(split_line)
    
        #header lines = 6 split values (6 displacement vectors per 'block')
        if num_elem_in_line == 6:
            #print(f'Separated Displacement Vectors: {line}')
            
            if current_section:
                sections.append(current_section)
    
            #start a new section beginning with this line
            current_section = [line]
            
        #displacement lines = 7 split values (an index int. followed by 6 floats
        elif num_elem_in_line == 7:
            if current_section:
                current_section.append(split_line[1:])
        #     else:
        #         # Handle case where a 7-element line appears before a 6-element start
        #         print(f"Warning: 7-element line found before a 6-element section start: {line}")
    
        # else: # Handle case where a 7-element line appears before a 6-element start
        #     print(f"Warning: 7-element line found before a 6-element section start: {line}")         
    
    if current_section:
        sections.append(current_section)
        
    # print(f"{output_file_path.name} contains {len(sections)} blocks of displacement vectors.")
    # print("")
    
    #make an empty df to store extracted displacement vectors
    displacement_df = pd.DataFrame()
    
    #
    for displacement_block in sections:
        #the first line of each extracted block contains the ID of the displacement vector
        #ex: the first line of the first block is '0 1 2 3 4 5' and corresponds to the 0, 1, 2nd 
        #displacement vectors
        block_first_line = displacement_block[0]
        #split the captured first line and store as column headers
        block_col_headers = list(map(int, block_first_line.split()))
    
        #make a block df using the extracted column headers and the add corresp. vector data beneath each
        block_df = pd.DataFrame(displacement_block[1:], columns=block_col_headers)
    
        #add the block's DF to the big DF containing all the displacement vectors
        displacement_df = pd.concat([displacement_df, block_df], axis=1)
    
    #this is the final displacement vector section as a pd dataframe
    return displacement_df

In [21]:
def write_resub_inp_for_failed_imag_freq(path_to_out_file: Path, path_to_resub_dir: Path):
    """
    Function:
    Inputs:
        -path_to_out_file
        -path_to_resub_dir
    Returns: 
        - 
    """
    #Specify a scaling factor (default is 0.1 right now)
    scaling_factor = 0.1

    file_name = path_to_out_file.stem

    #Get the last geom from .out file (has at least 1 IF)
    im_freq_geom = get_last_geom(path_to_out_file) #this is a pd DF

    #pull the raw freqs from the out file
    extracted_freqs = get_orca5_freqs(path_to_out_file)

    imag_indices = [i for i, freq in enumerate(extracted_freqs) if freq <0]
    # if imag_indices:
    #     print(f"\nIndices of imaginary freq(s): {imag_indices}")

    #in our present case I think/hope I can just adjust a single IF and resub
    if len(imag_indices) >= 1:
        imag_index = imag_indices[0]
    #need to add a way of handling multiple IFs/sorting by magnitude etc.
    #for now let's assume they have one/just use the first IF index

    #pull the displacement vector data from freq output
    displacement_df = extract_displacement_vectors_as_df(path_to_out_file)

    #get the displacement vector corresponding to the index of the imaginary freq
    imag_displacement_vector = displacement_df.iloc[:, imag_index]
    vector_list = imag_displacement_vector.tolist() #convert it to a list

    #convert the displacement vector list (1 col) to a 3 col numpy array (x y z): 
    three_col_displ_vector = np.array(vector_list).reshape(-1, 3).astype(float)

    #apply the specified scaling factor to the displacement array
    #this matrix determines how much each atom's coordinates will be moved by via addition
    displacement_scaled_array = np.multiply(three_col_displ_vector, scaling_factor)

    raw_geom =im_freq_geom.drop('AtomID', axis=1) #drop the atom ID from the input geom
    start_geom_array = raw_geom.to_numpy().astype(float) #convert the initial geom to a float array

    #apply the calculated displacement to the starting geom (make a new geom)
    new_geom_coords_array = start_geom_array + displacement_scaled_array
    df_from_numpy = pd.DataFrame(new_geom_coords_array) #convert the resulting np array to df

    #make a new df using OG atom list and the new xyz coords
    new_geom_df = im_freq_geom.iloc[:, :1] #get the atom ID col
    new_geom_df = pd.concat([new_geom_df, df_from_numpy ], axis=1) #add our newly calc. df 
    new_geom_df = np.round(new_geom_df, decimals = 6) #can round to diff. floats, here, 6)

    df_str = new_geom_df.astype(str) #df is now a string for writing
    new_geom_xyz = [' '.join(row) for row in df_str.values]

    ### Now have the new geometry; get the initial job params for resub
    inp_from_out = get_orca5_inp_from_out(path_to_out_file)
    
    #Rip out the current xyz coordinates from captured input:
    stripped_block = strip_geometry_from_inp_orca5(inp_from_out)

    #splice in our new geom:
    new_inp = splice_new_geom_into_inp(stripped_block, new_geom_xyz)
    #have to flatten the list into a \n-sep string for writing - annoying
    new_inp_as_str = '\n'.join(new_inp)
    new_inp_as_str += '\n'

    inp_file_name = file_name+".inp"
    write_path = path_to_resub_dir / inp_file_name
    write_path.parent.mkdir(parents=True, exist_ok=True) #make sure the dir exists
    write_path.write_text(new_inp_as_str, encoding='utf-8') #write the new .inp file
    print(f'New inp for resubmission of {file_name} written to /{path_to_resub_dir.stem}/')

In [22]:
def find_extra_inp_out_files_in_dirs(dir_with_out_files: Path, dir_with_inp_files: Path):
#This cell compares the filenames (no extension) of a dir with .out files and a dir with .inp files
#It will create and return a list of filenames that are in one directory but not the other
    print(f"\033[1mParsing specified .inp/.out directories for extra .inp/.out files....") #debug
    print(f"Directory with .out files: {dir_with_out_files}")
    print(f"Directory with .inp files: {dir_with_inp_files}\n")
       
    #dir_with_outs
    out_files = get_filepaths_in_target_dir_recursive(dir_with_out_files, ".out")
    print(f"\033[1mFound {len(out_files)} 'ORCA.out' DFT output files in /{dir_with_out_files.stem}/") #debug
    
    #dir_with_inps
    inp_files = get_filepaths_in_target_dir_recursive(dir_with_inp_files, ".inp")
    print(f"\033[1mFound {len(inp_files)} 'ORCA.inp' DFT input files in /{dir_with_inp_files.stem}/") #debug

    outs_no_ext = [] #stores the filenames of .outs in /out_dir
    for out_file in out_files:
        out_no_ext = out_file.stem
        outs_no_ext.append(out_no_ext)
        
    inps_no_ext = [] #stores the filenames of .inps in /inp_dir
    for inp_file in inp_files:
        inp_no_ext = inp_file.stem
        inps_no_ext.append(inp_no_ext)
    
    # print(inps_no_ext[:5]) #debug to check formatting
    # print(outs_no_ext[:5]) #debug to check formatting

    #give some info back to user about where the extra files are and calc. the set difference 
    if len(outs_no_ext) > len(inps_no_ext):
        difference_list = set(outs_no_ext) ^ set(inps_no_ext)
        if len(difference_list) > 0:
            print(f"\nYou are missing .inps; {len(difference_list)} Missing .inps!\nLengths: outs = {len(outs_no_ext)}, .inps = {len(inps_no_ext)}")

        else:
            print(f"\nYou have extra .outs; Make sure you remove any extraneous .outs from reruns;\n.outs= {len(outs_no_ext)}, .inps = {len(inps_no_ext)}")
        
    elif len(inps_no_ext) > len(outs_no_ext):
        difference_list = set(outs_no_ext) ^ set(inps_no_ext)
        print(f"\nMore .inps than .outs; {len(difference_list)} missing .outs!\nLengths: .outs = {len(outs_no_ext)}, .inps = {len(inps_no_ext)}")
        
    else:
        difference_list = set(outs_no_ext) ^ set(inps_no_ext)
        print(f"\nNumber of .inps and .outs match!\nLengths: .outs = {len(outs_no_ext)}, .inps = {len(inps_no_ext)}")
    
    if len(difference_list) > 0: 
        #print(f'Number of extra files: {len(difference_list)}')
        print(f'\nMissing File Names:')
        for file in difference_list:
            print(f'\t{file}')

        return difference_list

    elif len(difference_list) == 0:
        return False #can pass a false as there's no difference between sets

In [23]:
def find_and_del_replaced_outs(path_to_success_outs: Path, path_to_failed_outs: Path):
    """
    Function: Searches your /failed_dir/ for any .out files that have been replaced by valid .outs in /success.
        ex: you re-ran 100 jobs that had IFs (and hence were in /failed/IFs dir) and 99 of them were successful;
        After downloading the second-pass .outs, this method is used to clear the replaced jobs in /IF 
        
    Input:
        -path_to_out_files: path to a directory containing orca.out files
        -path_to_failed_files: path to a directory containing old/potentially outdated .out files
    Returns:
        - 
    """

    #specify a dir with good jobs and 1 with bad/failed jobs
    target_success_dir = path_to_success_outs
    parent_success = target_success_dir.parent
    parent_success_name = parent_success.name
    
    target_fail_dir = path_to_failed_outs

    #get all the failed jobs in /failed dir
    fail_outs = get_filepaths_in_target_dir_recursive(target_fail_dir, ".out")
    print(f'Found {len(fail_outs)} failed ORCA.out files in /{target_fail_dir.stem}/.')

    num_deleted = 0
    
    #loop through the list of failed jobs and see if the same job exists in the "good" dir
    for fail_out in fail_outs:
        fail_filename = fail_out.name #get just the extension
        #print(fail_filename)

        #create a dir to check /success with - does /success have the same file name (bad has been replaced)
        file_dir_to_check = target_success_dir / fail_filename
        #print(f"searching for: {file_dir_to_check} in /success_dir.")

        if file_dir_to_check.is_file(): #if the .out file exists in /success, 
            print(f"Found successful '{file_dir_to_check.name}' in /{parent_success_name}/. Deleting redundant .out in /{target_fail_dir.stem}/")
            try:
                fail_out.unlink(missing_ok=True) # delete the .out file in /fail (it's been fixed)
                num_deleted += 1
            except: 
                print(f"Error: File '{fail_out}' not found.")

    if num_deleted > 0:
        print(f"\nDeleted {num_deleted} extraneous .out files from {target_fail_dir}.")

In [24]:
### Perform Initial Pass/Fail Sort of aryne.outs ### 
"""
This cell performs a first pass/fail triage of any .out file in the targeted directory. It is looking
for 'Normal Termination' on the jobs as a minimum first pass - any failed convergence calcs are sent to
/failed_jobs for resubmission after generating a new .inp. Note - running this cell alone is not enough
to validate an .out file - you MUST verify (using the cell below) that the geometry corresponds to a 
minimum with no imaginary frequencies. 
"""

target_dir = aryne_dft_output_path
print(f"\033[1mPerforming initial Pass/Fail triage of .out files in '/{target_dir.stem}/\033[0m'...\n")

output_files = get_filepaths_in_target_dir_recursive(target_dir, ".out")

#Print the first 5 paths for debug
for out_file in output_files[:5]:
    print(out_file)
print(f"...\n")

### First Pass Triage of Outputs - Do they terminate normally, fail to reach convergence, or fail totally
for out_file in output_files:
    sort_complete_fail_jobs_orca5(out_file)

print(f"\033[1m\nFinished Pass/Fail triage of {len(output_files)} .out files in '/{target_dir.stem}/\033[0m'")

Performing initial Pass/Fail triage of .out files in '/Aryne_Orca_DFT_Outputs/'...

Found 9460 .out files in '/Aryne_Orca_DFT_Outputs/' (recursive)
C:\Chem_Work\CompChem_Working\CCAS_Hetarynes_Local\DFT_Aryne_Data\Aryne_Orca_DFT_Outputs\aryne_1_rdkit_conf_1.out
C:\Chem_Work\CompChem_Working\CCAS_Hetarynes_Local\DFT_Aryne_Data\Aryne_Orca_DFT_Outputs\aryne_2_rdkit_conf_1.out
C:\Chem_Work\CompChem_Working\CCAS_Hetarynes_Local\DFT_Aryne_Data\Aryne_Orca_DFT_Outputs\aryne_3_rdkit_conf_1.out
C:\Chem_Work\CompChem_Working\CCAS_Hetarynes_Local\DFT_Aryne_Data\Aryne_Orca_DFT_Outputs\aryne_4_rdkit_conf_1.out
C:\Chem_Work\CompChem_Working\CCAS_Hetarynes_Local\DFT_Aryne_Data\Aryne_Orca_DFT_Outputs\aryne_5_rdkit_conf_1.out
...

	'aryne_1_rdkit_conf_1.out': Potentially Valid
	'aryne_2_rdkit_conf_1.out': Potentially Valid
	'aryne_3_rdkit_conf_1.out': Potentially Valid
	'aryne_4_rdkit_conf_1.out': Potentially Valid
	'aryne_5_rdkit_conf_1.out': Potentially Valid
'aryne_6_rdkit_conf_1.out': 	Failed Conver

In [32]:
### Perform FREQ analysis on ARYNES ### 
#%%time

"""
This cell checks any .out that was deemed "potentially valid" by the preceding cell and checks the calc for
imaginary frequencies. Minima must have 0 imaginary frequencies, so if a job has one +, it is deemed a 
failure. These jobs can be reset, below, to try to remove the IF on a second pass. 
"""

target_dir = aryne_dft_output_path
print(f"\033[1mAnalyzing frequency data in potentially valid .out files in '/{target_dir.stem}/\033[0m'...\n")

#gather .out files in a target dir
output_files = get_filepaths_in_target_dir(target_dir, ".out")

#print out the first 5 jobs/paths for debug
for out_file in output_files[:5]:
    print(out_file)
print(f"...\n")

#place to shove bad if calcs
imag_freq_dir = aryne_im_freq_dir

#Loop over each .out file and determine if they have an IF or more
list_of_mol_freqs = []
for out_file in output_files:
    #check if the .out file contains any frequency data
    result = does_out_contain_freq_data(out_file)

    if result: #if true, file has frequency data in it
        extracted_freqs = get_orca5_freqs(out_file) #extract the frequency block

        #check if any of the extracted freqs are imaginary (condition of valid calc)
        freq_test_failed = has_imaginary_freqs_check(extracted_freqs)

        if freq_test_failed: #calculation has at least one IF (bad)
            #move the file to an error dir /no_freq_data/
            destination_file_path = imag_freq_dir / out_file.name
            try:
                shutil.move(out_file, destination_file_path)
                print(f"\t{out_file.name}: \tIMAGINARY FREQS! Moved to /Extra_Imaginary_Frequency/.")
            except FileNotFoundError:
                print(f"Error: Source file not found at {out_file.parent}")
            except shutil.SameFileError:
                print(f"\tSkipped '{out_file.name}', already exists in destination.")
            except Exception as e:
                print(f"\tError copying {out_file.name}: {e}")
                
        else: #No imaginary frequncy found in calc => it's probably great
            print(f"\t{out_file.name}: Valid Calc")

print(f"\033[1m\nCompleted analysis of frequency data for .out files in '/{target_dir.stem}/'\033[0m")

Analyzing frequency data in potentially valid .out files in '/Aryne_Orca_DFT_Outputs/'...

Found 8760 .out files in '/Aryne_Orca_DFT_Outputs/'
C:\Chem_Work\CompChem_Working\CCAS_Hetarynes_Local\DFT_Aryne_Data\Aryne_Orca_DFT_Outputs\aryne_1_rdkit_conf_1.out
C:\Chem_Work\CompChem_Working\CCAS_Hetarynes_Local\DFT_Aryne_Data\Aryne_Orca_DFT_Outputs\aryne_2_rdkit_conf_1.out
C:\Chem_Work\CompChem_Working\CCAS_Hetarynes_Local\DFT_Aryne_Data\Aryne_Orca_DFT_Outputs\aryne_3_rdkit_conf_1.out
C:\Chem_Work\CompChem_Working\CCAS_Hetarynes_Local\DFT_Aryne_Data\Aryne_Orca_DFT_Outputs\aryne_4_rdkit_conf_1.out
C:\Chem_Work\CompChem_Working\CCAS_Hetarynes_Local\DFT_Aryne_Data\Aryne_Orca_DFT_Outputs\aryne_5_rdkit_conf_1.out
...

	aryne_1_rdkit_conf_1.out: Valid Calc
	aryne_2_rdkit_conf_1.out: Valid Calc
	aryne_3_rdkit_conf_1.out: Valid Calc
	aryne_4_rdkit_conf_1.out: Valid Calc
	aryne_5_rdkit_conf_1.out: Valid Calc
	aryne_8_rdkit_conf_1.out: Valid Calc
	aryne_9_rdkit_conf_1.out: Valid Calc
	aryne_10_rdkit_

In [33]:
### Get some analytics on pass/fail after a big pass of DFT calculations ### 
"""
This cell is for analysis on job completion/failures. It will report how many jobs are still "successful" and
also tell you the # of failures based on how they were triaged. Ideally you should have all of your .outs accounted
for, but the cells below this one can help find missing .inps/.outs. The next step after this is to reset any
failed .outs/resubmit jobs as desired. 
"""

#Directories that contain .inps, "good".outs, and fail.outs, respectively
target_inp_dir = aryne_dft_input_path
target_out_dir = aryne_dft_output_path
target_fail_dir = aryne_failed_to_terminate

#how many .inp files in /target_inp_dir?
input_filepaths = get_filepaths_in_target_dir_recursive(target_inp_dir, ".inp")
num_expected = len(input_filepaths)
print(f"\033[1mFound {num_expected} 'ORCA.inp' DFT input files in '{target_inp_dir.stem}' => Expected ORCA.outs: {num_expected}\n\033[0m") #debug

#get a list of the validated .outs in output directory
#MAKE SURE YOU HAVE PROCESSED YOUR DATA USING THE ABOVE 2 CELLS! Or I'll haunt you 0.0
validated_out_filepaths = get_filepaths_in_target_dir(target_out_dir, ".out")

#Gather all the failed .outs (recursive, so will get all files in /target_fail_dir/subdir(s))
failed_out_filepaths = get_filepaths_in_target_dir_recursive(target_fail_dir, ".out")

#how many total good and failed .outs are in the /target_out_dir
accounted_for_filepaths = list(validated_out_filepaths + failed_out_filepaths)
num_outs_accounted_for = len(accounted_for_filepaths)

#gives some extra details
how_many_jobs_where(target_out_dir)

Found 10123 .inp files in '/Aryne_Orca_DFT_Inputs/' (recursive)
Found 10123 'ORCA.inp' DFT input files in 'Aryne_Orca_DFT_Inputs' => Expected ORCA.outs: 10123

Found 8760 .out files in '/Aryne_Orca_DFT_Outputs/'
Found 700 .out files in '/Failed_Calculations/' (recursive)

Checking C:\Chem_Work\CompChem_Working\CCAS_Hetarynes_Local\DFT_Aryne_Data\Aryne_Orca_DFT_Outputs for Orca.out files after processesing...

Number of VALIDATED ORCA.out files in Aryne_Orca_DFT_Outputs: 8760

Number of FAILED ORCA.outs in /Failed_Calculations/:          700
	- Failed due to extraneous Imaginary Freqs:           288
	- Failed due to lack of convergence:                  375
	- Complete and Utter Hopeless Failures:                37
	- Failed due to missing freq data:                      0

Total number of ORCA.out files in /Aryne_Orca_DFT_Outputs/ accounted for: 9460


In [27]:
### DO YOUR .OUTs =/= YOUR .INPs? Use this cell/method to find the offending file names ### 
"""
Ex: 
    - you ran a first round of DFT calculations and you don't have all your .outs (missing .outs)
    - you find out you are missing .inp files somehow and want to find out which to regen/find
"""
#(dir containing your .out files, dir containing your .inp files
difference_found = find_extra_inp_out_files_in_dirs(aryne_dft_output_path, aryne_dft_input_path)

if difference_found: #this will print an error message if you are missing .inp files
    print("")
    print("Look for your .inp file(s)!")

Parsing specified .inp/.out directories for extra .inp/.out files....
Directory with .out files: C:\Chem_Work\CompChem_Working\CCAS_Hetarynes_Local\DFT_Aryne_Data\Aryne_Orca_DFT_Outputs
Directory with .inp files: C:\Chem_Work\CompChem_Working\CCAS_Hetarynes_Local\DFT_Aryne_Data\Aryne_Orca_DFT_Inputs

Found 9460 .out files in '/Aryne_Orca_DFT_Outputs/' (recursive)
Found 9460 'ORCA.out' DFT output files in /Aryne_Orca_DFT_Outputs/
Found 9460 .inp files in '/Aryne_Orca_DFT_Inputs/' (recursive)
Found 9460 'ORCA.inp' DFT input files in /Aryne_Orca_DFT_Inputs/

Number of .inps and .outs match!
Lengths: .outs = 9460, .inps = 9460


In [28]:
### DO YOU HAVE EXTRA .OUTS? Use this cell/method ###
"""
#Ex: you used this notebook to regenerate new .inp files for failed .outs in /convergence_failure 
    #and/or #/extra_imaginary_frequency. You then re-ran those jobs and downloaded the new .outs 
    #to your /out_files_dir. You've parsed the .outs but now you have too many. There are likely 
    #extra .outs in your failed_dir that need to be deleted since now the job has terminated without 
    #an #error/has been deemed a success. This method will find and delete and extra failed .outs.
"""
#path to your successful .outs, path to your failed .outs
find_and_del_replaced_outs(aryne_dft_output_path, aryne_failed_to_terminate)
#check_for_replaced_outs(aryne_dft_output_path, aryne_failed_to_terminate)

Found 700 .out files in '/Failed_Calculations/' (recursive)
Found 700 failed ORCA.out files in /Failed_Calculations/.


In [29]:
### RESET FAILED.OUTs in /failed_convergence/ - WRITE NEW .INPs for Resubmission on HPC ### 
"""
    Run this cell to generate new .inp files for any jobs currently in the /failed_calculation/..
    ../convergence_failure/ directory. New inputs will be written to input/resub by default, here. 
    New input files are generated using the last geometry extracted from the .out file and reads 
    the input details/level of theory from the .out file. 
"""
print(f'Generating new .inp files for jobs that failed to converge.\n')

#point at the correct dirs
failed_conv_dir = aryne_no_convergence_dir
resub_dir = aryne_dft_resub_input

#grab the files in the failed convergence directory
fail_conv_filepaths = get_filepaths_in_target_dir(failed_conv_dir, ".out")
print(f'Located {len(fail_conv_filepaths)} .out files to make new .inps for.\n')

write_count = 0 #number of files written to /resub

#Reset just pulls the last geom from the .out file
### To-Do - some inferencing on prior job outcome => better input handling
for bad_opt in fail_conv_filepaths:
    #pulls inp data from .out file; pulls last geometry from prev. opt; gens new .inp 
    #with same LOT and writes to a specified dir. 
    write_resub_inp_for_failed_conv(bad_opt, resub_dir)
    write_count +=1 #inc the counter 

print(f"\nWrote {write_count} new .inp files for resubmission to {resub_dir}")

Generating new .inp files for jobs that failed to converge.

Found 375 .out files in '/Convergence_Failure/'
Located 375 .out files to make new .inps for.

New inp for resubmission of aryne_6_rdkit_conf_1 written to /Aryne_Resubs/
New inp for resubmission of aryne_7_rdkit_conf_1 written to /Aryne_Resubs/
New inp for resubmission of aryne_28_rdkit_conf_1 written to /Aryne_Resubs/
New inp for resubmission of aryne_31_rdkit_conf_1 written to /Aryne_Resubs/
New inp for resubmission of aryne_34_rdkit_conf_1 written to /Aryne_Resubs/
New inp for resubmission of aryne_66_rdkit_conf_1 written to /Aryne_Resubs/
New inp for resubmission of aryne_74_rdkit_conf_1 written to /Aryne_Resubs/
New inp for resubmission of aryne_157_rdkit_conf_1 written to /Aryne_Resubs/
New inp for resubmission of aryne_187_rdkit_conf_1 written to /Aryne_Resubs/
New inp for resubmission of aryne_202_rdkit_conf_1 written to /Aryne_Resubs/
New inp for resubmission of aryne_348_rdkit_conf_1 written to /Aryne_Resubs/
New in

In [30]:
### Resetting /extra_imaginary_frequency/ jobs: Write new .inp files for jobs that failed due to IF(s) ### 
print(f'Generating new .inp files for jobs that failed due to having imaginary frequency/ies.\n')

"""
    Run this cell to generate new .inp files for any jobs currently in the /failed_calculation/..
    ../extra_imaginary_frequency directory. New inputs will be written to input/resub by default, here. 
    New geometries are generated by displacing geometries along an imaginary frequency using the norm-
    alized displacement vector and a 0.15 Ang. scalar. 
"""

target_dir = aryne_im_freq_dir
resub_dir = aryne_dft_resub_input

filepaths = get_filepaths_in_target_dir(target_dir, ".out")
print(f'Located {len(filepaths)} .out files to make new .inps for.\n')

write_count = 0 #number of files written to /resub

#calculate a new inp. geometry for resub
#uses the last geom in the .inp file in a dumb way so these can still fail to opt
for if_opt in filepaths:
    write_resub_inp_for_failed_imag_freq(if_opt, resub_dir)
    write_count +=1 #inc the counter 

print(f"\nWrote {write_count} new .inp files for resubmission to {resub_dir}")

Generating new .inp files for jobs that failed due to having imaginary frequency/ies.

Found 288 .out files in '/Extra_Imaginary_Frequency/'
Located 288 .out files to make new .inps for.

New inp for resubmission of aryne_40_rdkit_conf_1 written to /Aryne_Resubs/
New inp for resubmission of aryne_91_rdkit_conf_1 written to /Aryne_Resubs/
New inp for resubmission of aryne_149_rdkit_conf_1 written to /Aryne_Resubs/
New inp for resubmission of aryne_391_rdkit_conf_1 written to /Aryne_Resubs/
New inp for resubmission of aryne_395_rdkit_conf_1 written to /Aryne_Resubs/
New inp for resubmission of aryne_423_rdkit_conf_1 written to /Aryne_Resubs/
New inp for resubmission of aryne_432_rdkit_conf_1 written to /Aryne_Resubs/
New inp for resubmission of aryne_466_rdkit_conf_1 written to /Aryne_Resubs/
New inp for resubmission of aryne_510_rdkit_conf_1 written to /Aryne_Resubs/
New inp for resubmission of aryne_528_rdkit_conf_1 written to /Aryne_Resubs/
New inp for resubmission of aryne_638_rdkit_

In [31]:
resub_dir = aryne_dft_resub_input
split_dir_into_subdirs(resub_dir, 50)

Found 663 DFT Orca.inp files in C:\Chem_Work\CompChem_Working\CCAS_Hetarynes_Local\DFT_Aryne_Data\Aryne_Orca_DFT_Inputs\Aryne_Resubs
